
#3D point-cloud registration

In this notebook we are going to implement the full process of 3D rigid registration of two point-clouds. 3D registration refers to find the transformation that best aligns two point-clouds, and in this case we will focuse on rigid transformations.

## Registration pipeline

The point-cloud registration (PCR) is defined by the following equation:



> $T* = \underset{T \in \tau}{\mathrm{argmin}} \sum_{i,j}^{N,K} w_{i,j} \| t-sRm_i - s_j\|$



, where $m_i \in \mathbb{R}^{3 \times N}$ are the points in the model *M* that will be mapped onto the source *S* compound by  $s_j \in \mathbb{R}^{3 \times K}$ 3D points. $w_{i,j}$ is the similarity of the pair of points $\{m_i, s_j\}$ taken into account. In the case of binary correspondences where one-to-one matching is consiered:

> $w_{i,j} = \begin{cases}
    1 ,& m_i ~best~match~ s_j\\
    0,              & \text{otherwise}
\end{cases}$





---



The PCR pipeline is mainly divided intro three stages:

1. Feature extraction for pairing
2. Matching
3. Transformation estimation

In this small project we will apply the pipeline in a coarse RANSAC-based registration method and after in a fine ICP-like approach. Finally, both implementations will work together to achieve the best alignment between two point-clouds.


In [ ]:
!pip install -q open3d

In [ ]:
import open3d as o3d
import numpy as np
from scipy.spatial.transform import Rotation as R
import copy
from sklearn.neighbors import NearestNeighbors
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import random

In [ ]:
def plotCorrespondences(pointCloud1, pointCloud2, correspondences):
  # Create lines between corresponding points
  lines = o3d.utility.Vector2iVector([[corr[0], corr[1]] for corr in correspondences])

  # Create line set
  line_set = o3d.geometry.LineSet()
  line_set.lines = lines
  o3d.visualization.draw_plotly([pointCloud1, pointCloud2, line_set])


In [ ]:
def plotCorrespondences2(pcd1, pcd2, correspondences):
    fig = plt.figure(figsize=(10, 5))
    pointCloud1= np.asarray(pcd1.points)
    pointCloud2= np.asarray(pcd2.points)

    # Plot correspondences
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.scatter(pointCloud1[:, 0], pointCloud1[:, 1], pointCloud1[:, 2], c='b', label='PointCloud1')
    ax2.scatter(pointCloud2[:, 0], pointCloud2[:, 1], pointCloud2[:, 2], c='r', label='PointCloud2')
    for corr in correspondences:
        p1 = pointCloud1[corr[0]]
        p2 = pointCloud2[corr[1]]
        ax2.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], c='g', linewidth=0.5)
    ax2.set_title('Correspondences')

    plt.tight_layout()
    plt.show()

In [ ]:
def visualizar_correspondencias(puntos_src, puntos_dst, correspondencias):
    """
    Visualiza las correspondencias entre dos nubes de puntos.

    Parámetros:
    - puntos_src (np.ndarray): Arreglo Numpy con las coordenadas de la nube de puntos fuente.
    - puntos_dst (np.ndarray): Arreglo Numpy con las coordenadas de la nube de puntos destino.
    - correspondencias (np.ndarray): Arreglo de índices que indican las correspondencias entre puntos.
                                      Por ejemplo, correspondencias[i] = j indica que el punto i en
                                      puntos_src corresponde al punto j en puntos_dst.
    """

    # Crear figura
    fig = go.Figure()

    # Agregar nube de puntos fuente
    fig.add_trace(go.Scatter3d(x=puntos_src[:, 0], y=puntos_src[:, 1], z=puntos_src[:, 2],
                               mode='markers', marker=dict(size=2, color='blue'),
                               name='Fuente'))

    # Agregar nube de puntos destino
    fig.add_trace(go.Scatter3d(x=puntos_dst[:, 0], y=puntos_dst[:, 1], z=puntos_dst[:, 2],
                               mode='markers', marker=dict(size=2, color='red'),
                               name='Destino'))

    # Agregar líneas de correspondencia
    for i, j in enumerate(correspondencias):
        fig.add_trace(go.Scatter3d(x=[puntos_src[i, 0], puntos_dst[j, 0]],
                                   y=[puntos_src[i, 1], puntos_dst[j, 1]],
                                   z=[puntos_src[i, 2], puntos_dst[j, 2]],
                                   mode='lines', line=dict(color='green', width=2),
                                   showlegend=False))

    # Actualizar la disposición de la figura para una mejor visualización
    fig.update_layout(scene=dict(
                        xaxis_title='X AXIS',
                        yaxis_title='Y AXIS',
                        zaxis_title='Z AXIS'),
                      width=700,
                      margin=dict(r=20, b=10, l=10, t=10))
    fig.show()

# Initialization

Initially we acquire or read two point clouds. However, for the sake of evaluation, we create a copy of one point cloud and transform it with a known rotation and translation (ground truth). Hence, we can compare the ground truth with the transformation that we infer using the registration method.  


In [ ]:
# Import point cloud
pointCld = o3d.data.DemoICPPointClouds()
source = o3d.io.read_point_cloud(pointCld.paths[0])
#target = o3d.io.read_point_cloud(pointCld.paths[1])

# Generate known transformation (Ground Truth)
r = R.from_quat([0, 0, np.sin(np.pi/4), np.cos(np.pi/4)])
RotationVector = r.as_euler('zyx', degrees=True)
RotationMatrix = r.as_matrix()
TranslationVector = [1, 1.3, 0]

GT = np.eye(4)
GT[:3, :3] = RotationMatrix
GT[:3,3] = TranslationVector

# print(GT)
# Generate known target by transforming source
target = copy.deepcopy(source).transform(GT)



---
# Feature-based
For this example, we are going to use FPFH 3D features.

URL: https://www.open3d.org/docs/latest/python_api/open3d.registration.compute_fpfh_feature.html

The main steps will be:
- Downsample the point clouds: Since the registration is rigid, all points in the same point cloud will suffer the same transformation. Hence, and in the sake of unburdening the processor, we will reduce the amount of data by applying a downsampling.
- Feature extraction: with the downsampled data sets, features are estimated. In this case, we will use the FPFH that is include in the Open3D library.
- Find the best match: compare the features in both data sets and estimate which are the closest.  
- Reject outliers. **Ransac**:
  - Get a random subsample of matches
  - Optimize the transformation using Singular Value Decomposition (SVD)
  - Check the matches that are within a threshold (inliers) and which are outside (outliers)
  - Repeat N times.
  - Keep the transformation that had more inliers.

In [ ]:
# Subsampling the point clouds
voxelSize = 0.05
sourceDown = source.voxel_down_sample(voxel_size = voxelSize)
targetDown = target.voxel_down_sample(voxel_size = voxelSize)
print("Original number of points: " + str(np.asarray(source.points).shape))
print("Sampled number of points: " + str(np.asarray(sourceDown.points).shape))

# Plot point clouds
o3d.visualization.draw_plotly([sourceDown, targetDown])

In [ ]:
# Feature extraction - FPFH 3D features
voxelSize = 0.05
sourceDown.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius = voxelSize*2, max_nn=30))
targetDown.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius = voxelSize*2, max_nn=30))

Sourcefpfh = o3d.pipelines.registration.compute_fpfh_feature(sourceDown, o3d.geometry.KDTreeSearchParamHybrid(radius=voxelSize*5, max_nn=100))
Targetfpfh = o3d.pipelines.registration.compute_fpfh_feature(targetDown, o3d.geometry.KDTreeSearchParamHybrid(radius=voxelSize*5, max_nn=100))

In [ ]:
# Find the best match
k = 1
knn = NearestNeighbors(n_neighbors=k)
knn.fit(Sourcefpfh.data.T)
NearestNeighbors(algorithm='auto', leaf_size=30, n_neighbors=k, p=2, radius=1.0)
TargetNeighbor = knn.kneighbors(Targetfpfh.data.T, return_distance=False)

# Reorder points so we have the matches aligned
reorderingIndices = TargetNeighbor.squeeze().astype(int)
rearrangedPoints = np.asarray(sourceDown.points)[reorderingIndices]
rearrangedNormals = np.asarray(sourceDown.normals)[reorderingIndices]
rearrangedColor = np.asarray(sourceDown.colors)[reorderingIndices]

sourceReordered = copy.deepcopy(sourceDown)
sourceReordered.points = o3d.utility.Vector3dVector(rearrangedPoints)
sourceReordered.normals = o3d.utility.Vector3dVector(rearrangedNormals)
sourceReordered.colors = o3d.utility.Vector3dVector(rearrangedColor)

In [ ]:
# RANSAC
valueRange = np.arange(0, np.asarray(sourceDown.points).shape[0], 1)
nIters = 10
subset_size = 15

# Initialize the subsetIndex array
subsetIndex = np.zeros((nIters, subset_size), dtype=int)

for x in range(nIters):
  subsetIndex[x,:] = random.sample(list(valueRange), 15)

  sourceSubset = np.asarray(sourceReordered.points)[subsetIndex[x,:]]
  targetSubset = np.asarray(targetDown.points)[subsetIndex[x,:]]

  sourceTransposed = sourceSubset.transpose()
  targetTransposed = targetSubset.transpose()

  # find the rotation and translation using procrusters and SVD

  sourceTransformed = copy.deepcopy(sourceSubset).transform(T)

  ## Estimate inliers
  # Define threshold
  # Calculate distance between each point pair of the transformed source and sampled target
  # Check the point pairs within the threshols
  # If the number of inliers > max inliers, keep that transformation



---

## Iterative Closest Point

The next step is to evaluate a fine grain registration method. In this case we will use ICP.

The process could be:
- Scan (load) two point clouds, or use the same technique as before and create a copy that is transformed using a Ground Truth transformation.
- Apply ICP
- Evaluate the result


In [ ]:
# Apply ICP


In [ ]:
# Combine RANSAC + ICP to achieve the best result




---
# Evaluation (out of 10 points)
- Complete the code above to make RANSAC work (2.5 points).
- Implement ICP (2.5 points)
- Combine RANSAC + ICP (0.5 points)
- Add noise to the point clouds using different levels of perturbation and evaluate the system (1 point).
- Reduce the initial subset of points in RANSAC and see what is the minimum number of data to obtain good results (1.5 points).
- In ICP, change the ground truth transformation with larger and smaller rotations and see when it fails to find the transformation (2 points).

## Go further
- If you want to go further in your knowledge, try Coherent Point Drift (CPD) non-rigid registration algorithm to align deformable objets (+1 extra point)
- Link: https://siavashk.github.io/2017/05/14/coherent-point-drift/
- !pip install -q probreg
- Test:
    - import probreg
    - result = probreg.cpd.registration_cpd()
